In [1]:
!pip install kagglehub==1.0.0

### Files Setup

In [15]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [16]:
try:
    path = Path('/content/drive/MyDrive/Universidade/Mestrado/2 Semestre/Aprendizagem Computacional Avançada/AML_Butterfly/aca-butterflies')

    if not path.exists():
        raise FileNotFoundError(f"The specified path does not exist: {path}")

except FileNotFoundError:
    path = Path('/content/drive/MyDrive/AdvancedMachineLearning2/aca-butterflies')

if not path.exists():
    raise FileNotFoundError(f"The specified path does not exist: {path}")

print(f"Using path: {path}")

FileNotFoundError: The specified path does not exist: /content/drive/MyDrive/AdvancedMachineLearning2/aca-butterflies

### Load data

In [17]:
# Load the dataset on Colab machine
!cp "/content/drive/MyDrive/Universidade/Mestrado/2 Semestre/Aprendizagem Computacional Avançada/AML_Butterfly/aca-butterflies.zip" "/content/butterflies.zip"

cp: cannot stat '/content/drive/MyDrive/Universidade/Mestrado/2 Semestre/Aprendizagem Computacional Avançada/AML_Butterfly/aca-butterflies.zip': No such file or directory


In [18]:
!cp "/content/drive/MyDrive/AdvancedMachineLearning2/aca-butterflies.zip" "/content/butterflies.zip"

In [19]:
!unzip -q /content/butterflies.zip -d /content/dataset_local

In [2]:
import os

path = None

if os.path.exists("aca-butterflies"):
    path = "./aca-butterflies"
else:
    # login in a different cell to avoid a known kagglehub issue
    import kagglehub
    kagglehub.login()

In [3]:
if path is None:
    # do not forget to enter the Kaggle competition through the link in the project statement (section 4)
    path = kagglehub.competition_download('aca-tp-2')
    print("Path to dataset files:", path)

UnauthenticatedError: User is not authenticated

In [4]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
import torch.optim as optim

In [5]:
BATCH_SIZE = 32
IMAGE_SIZE = 64

In [6]:
class ButterflyDataset(data.Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.img_labels = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

        self.classes = sorted(self.img_labels['label'].unique())
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_name = self.img_labels.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        label_name = self.img_labels.iloc[idx]['label']
        label_idx = self.class_to_idx[label_name]
        label = torch.tensor(label_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

## First, we read the dataset, preprocess the images and encapsulate them into dataloader form.

In [7]:
# load the data
img_dir = os.path.join(path, 'train')
df = pd.read_csv(os.path.join(path, 'train.csv'))

# preprocessing
data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# encapsulate data into dataloader form
dataset = ButterflyDataset(df=df, img_dir=img_dir, transform=data_transform)
dataloader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
print(f"Number of samples: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")

In [ ]:
# Get a batch of images from the dataloader
images, labels = next(iter(dataloader))

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()

for i, img in enumerate(images):
    img = img.permute(1, 2, 0).numpy()
    label_idx = labels[i].item()
    label_name = dataset.classes[label_idx]
    axes[i].imshow(img)
    axes[i].set_title(label_name.capitalize(), fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Classifier

In [20]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes):

        """
        Baseline Convolutional Neural Network for Butterfly Classification.
        
        Args:
            num_classes (int): Number of target species/classes (75 for this dataset).
        """

        super().__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                # Layer 1: Convolutional feature extraction preserving spatial dimensions
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                
                # Layer 2: Second convolution to build deeper feature representations
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),

                # Downsamples spatial dimensions by 2x (Height and Width halved)
                nn.MaxPool2d(2, 2)
            )

        # Feature Extractor: Progressively extracts high-level semantic abstractions
        self.features = nn.Sequential(
            conv_block(3, 64), # Input: 3 channels (RGB) -> Output: 64 channels
            conv_block(64, 128), # Input: 64 channels      -> Output: 128 channels
            conv_block(128, 256), # Input: 128 channels     -> Output: 256 channels
            # Global pooling forces spatial dimensions to 1x1, minimizing dense layer parameters
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Fully Connected Classifier: Maps extracted features to final class logits
        self.classifier = nn.Sequential(
            *[layer for size in [256, 512, 1024]
            for layer in (nn.Linear(size, size*2), nn.ReLU(inplace=True), nn.Dropout(p=0.5))],
            # Final output layer: Maps 2048 hidden units to the 75 butterfly classes
            nn.Linear(2048, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [ ]:
n_classes = len(dataset.classes)
model = BaselineCNN(num_classes=n_classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()